# PWV00 : Spectra Count Table — Selection Efficiency

## Goal

Build a comprehensive **selection efficiency table** for AuxTel/Spectractor spectra.
For each combination of:
- **Dataset epoch** : full dataset *vs.* post-collimator only (after 2023-09-30),
- **Filter subset** : all PWV filters (`empty`, `OG550_65mm_1`, `SDSSr`, `FELH0600`) *vs.* `empty` only *vs.* `OG550_65mm_1` only,
- **Cut level** : no cuts, standard cuts (`cuts_finaldecision.json`), loose cuts (`cuts_loose_finaldecision.json`), tight cuts (`cuts_tight_finaldecision.json`),

the notebook reports:
1. **N_total** — number of spectra before quality cuts,
2. **N_selected** — number of spectra surviving the quality cuts,
3. **efficiency** — $\varepsilon = N_{\rm selected} / N_{\rm total}$ (%).

The resulting table is displayed inline as a styled pandas DataFrame **and** exported as a ready-to-use **LaTeX table**.

- **Author :** Sylvie Dagoret-Campagne
- **Affiliation :** IJCLab/IN2P3/CNRS
- **Creation date :** 2026-03-21
- **Run version :** `run2026_v02b_cr_run2026_v02d_cr`
- **Kernel :** `conda_py313`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.resetwarnings()
warnings.simplefilter('ignore')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from astropy.time import Time

# ── Display options ───────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:.1f}'.format)

In [ ]:
from PWV00_parameters import (
    version_run, tag, extractedfilesdict,
    PWV_FILTER_LIST, FLAG_PWVFILTERS,
    DumpConfig
)
from mysitcom.auxtel.qualitycuts import ParameterCutSelection, ParameterCutTools
from mysitcom.auxtel.pwv import normalize_column_data_bytarget_byfilter

DumpConfig()

---
## 1. Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
pathdata = "data_PWV01seas"   # where the JSON cut files are stored
os.makedirs(pathdata, exist_ok=True)

# ── Collimator installation date ──────────────────────────────────────────────
# After this date D2CCD measurements are more repeatable.
DATETIME_COLLIMATOR = pd.to_datetime("2023-09-30 00:00:00+0000", utc=True)

# ── Cut-file definitions ──────────────────────────────────────────────────────
CUT_FILES = {
    "standard" : f"{pathdata}/cuts_finaldecision.json",
    "loose"    : f"{pathdata}/cuts_loose_finaldecision.json",
    "tight"    : f"{pathdata}/cuts_tight_finaldecision.json",
}

# ── Filter subsets ────────────────────────────────────────────────────────────
FILTER_SUBSETS = {
    "all PWV filters"  : PWV_FILTER_LIST,           # ["empty","OG550_65mm_1","SDSSr","FELH0600"]
    "empty"            : ["empty"],
    "OG550_65mm_1"     : ["OG550_65mm_1"],
}

print("PWV_FILTER_LIST :", PWV_FILTER_LIST)
print("Cut files       :", list(CUT_FILES.keys()))
print("Filter subsets  :", list(FILTER_SUBSETS.keys()))

---
## 2. Load and prepare the data

In [ ]:
atmfilename   = extractedfilesdict[version_run]
inputfilename = atmfilename.split("/")[-1]
print(f"Loading: {atmfilename}")

if "parquet" in inputfilename:
    df_raw = pd.read_parquet(atmfilename)
else:
    specdata = np.load(atmfilename, allow_pickle=True)
    df_raw   = pd.DataFrame(specdata)

# ── Parse time ────────────────────────────────────────────────────────────────
df_raw["Time"] = pd.to_datetime(df_raw["DATE-OBS"], utc=True)
df_raw = df_raw.sort_values("Time", ascending=True).reset_index(drop=True)

# ── Rename Corentin-run columns (chi2) ────────────────────────────────────────
if "run2026_v02" in version_run:
    df_raw["chi2_ram"] = df_raw["CHI2_FIT"]
    df_raw["chi2_rum"] = df_raw["CHI2_FIT"]

print(f"Raw shape: {df_raw.shape}")
print(f"Date range: {df_raw['Time'].min().date()}  →  {df_raw['Time'].max().date()}")
print("Available filters:", sorted(df_raw["FILTER"].unique()))

In [ ]:
# ── Keep only PWV-relevant filters ───────────────────────────────────────────
df_pwv = df_raw[df_raw["FILTER"].isin(PWV_FILTER_LIST)].copy()

# Drop rows with missing PWV columns
pwv_cols = ["PWV [mm]_ram", "PWV [mm]_rum",
            "PWV [mm]_err_ram", "PWV [mm]_err_rum"]
df_pwv.dropna(subset=pwv_cols, inplace=True)
df_pwv.reset_index(drop=True, inplace=True)

print(f"After PWV filter selection: {df_pwv.shape}")
print("Filters kept:", sorted(df_pwv["FILTER"].unique()))

In [ ]:
# ── Normalise chi² columns (required by ParameterCutSelection) ────────────────
for col in ["CHI2_FIT", "chi2_ram", "chi2_rum"]:
    df_pwv, _ = normalize_column_data_bytarget_byfilter(
        df_pwv,
        target_col="TARGET", filter_col="FILTER",
        feature_col=col, ext="norm"
    )

print("Normalised columns added.")
print("Total rows ready for quality cuts:", len(df_pwv))

---
## 3. Helper function — apply quality cuts to a sub-dataframe

In [ ]:
def apply_cuts(df: pd.DataFrame, cut_file: str) -> pd.DataFrame:
    """
    Apply quality cuts defined in *cut_file* (JSON) to *df*.
    Returns the sub-dataframe of rows that pass all cuts.

    Parameters
    ----------
    df        : DataFrame that has already been chi2-normalised.
    cut_file  : path to a JSON cuts file (ParameterCutTools format).

    Returns
    -------
    df_keep   : DataFrame of rows that pass all cuts.
    """
    cuts           = ParameterCutTools.load_cuts_json(cut_file)
    list_of_params = list(cuts.keys())
    selector       = ParameterCutSelection(df, params=list_of_params, id_col="id")
    flags          = selector.apply_cuts(cuts)
    df_merged      = df.merge(flags, on="id")
    df_keep        = df_merged[df_merged["pass_all_cuts"]].copy()
    return df_keep

---
## 4. Build the selection table

We iterate over all combinations of:
- epoch (`full` / `post-collimator`)
- filter subset
- cut level (`no cuts` / `standard` / `loose` / `tight`)

and collect `N_total`, `N_selected`, and `efficiency (%)` for each row.

In [ ]:
rows = []

epoch_defs = {
    "Full dataset"        : df_pwv,
    "Post-collimator\n(after 2023-09-30)" : df_pwv[df_pwv["Time"] > DATETIME_COLLIMATOR].copy(),
}

for epoch_label, df_epoch in epoch_defs.items():
    for filt_label, filt_list in FILTER_SUBSETS.items():

        # Filter selection
        df_sub = df_epoch[df_epoch["FILTER"].isin(filt_list)].copy()
        if len(df_sub) == 0:
            continue

        N_total = len(df_sub)

        # ── No cuts ──────────────────────────────────────────────────────────
        rows.append({
            "Epoch"         : epoch_label,
            "Filter subset" : filt_label,
            "Cut level"     : "no cuts",
            "N_total"       : N_total,
            "N_selected"    : N_total,
            "Efficiency (%)" : 100.0,
        })

        # ── Quality cut levels ────────────────────────────────────────────────
        for cut_label, cut_file in CUT_FILES.items():
            df_keep    = apply_cuts(df_sub, cut_file)
            N_selected = len(df_keep)
            efficiency = 100.0 * N_selected / N_total if N_total > 0 else 0.0
            rows.append({
                "Epoch"         : epoch_label,
                "Filter subset" : filt_label,
                "Cut level"     : cut_label,
                "N_total"       : N_total,
                "N_selected"    : N_selected,
                "Efficiency (%)" : efficiency,
            })

df_table = pd.DataFrame(rows)
print(f"Table shape: {df_table.shape}")
df_table

---
## 5. Display — styled DataFrame

In [ ]:
from IPython.display import display

def highlight_efficiency(val):
    """Colour-code efficiency: green ≥ 70 %, yellow 40–70 %, red < 40 %."""
    if not isinstance(val, float):
        return ''
    if val >= 70.0:
        return 'background-color: #c6efce; color: #276221'   # green
    elif val >= 40.0:
        return 'background-color: #ffeb9c; color: #9c6500'   # yellow
    else:
        return 'background-color: #ffc7ce; color: #9c0006'   # red


styled = (
    df_table
    .style
    .format({"N_total": "{:,d}", "N_selected": "{:,d}", "Efficiency (%)": "{:.1f}"})
    .applymap(highlight_efficiency, subset=["Efficiency (%)"])
    .set_caption(
        f"AuxTel/Spectractor — Spectra count and selection efficiency\n"
        f"Run: {version_run} | {tag}"
    )
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '13px'), ('font-weight', 'bold'),
                   ('text-align', 'left'), ('padding-bottom', '6px')]},
        {'selector': 'th',
         'props': [('background-color', '#4472C4'), ('color', 'white'),
                   ('font-size', '12px'), ('text-align', 'center')]},
        {'selector': 'td',
         'props': [('font-size', '12px'), ('text-align', 'center'),
                   ('padding', '4px 10px')]},
    ])
)

display(styled)

---
## 6. Pivot view — efficiency matrix

A condensed 2-D pivot table: rows = (Epoch, Filter subset), columns = Cut level.

In [ ]:
# ── N_selected pivot ──────────────────────────────────────────────────────────
pivot_n = df_table.pivot_table(
    index=["Epoch", "Filter subset"],
    columns="Cut level",
    values="N_selected",
    aggfunc="first"
)

# Reorder columns
col_order = [c for c in ["no cuts", "loose", "standard", "tight"] if c in pivot_n.columns]
pivot_n = pivot_n[col_order]

# ── Efficiency pivot ──────────────────────────────────────────────────────────
pivot_eff = df_table.pivot_table(
    index=["Epoch", "Filter subset"],
    columns="Cut level",
    values="Efficiency (%)",
    aggfunc="first"
)[col_order]

print("=== N_selected ===")
display(pivot_n)
print("\n=== Efficiency (%) ===")
display(pivot_eff.style.format("{:.1f}").background_gradient(cmap='RdYlGn', axis=None, vmin=0, vmax=100))

---
## 7. LaTeX table

The cell below generates a complete LaTeX `longtable` that can be pasted
directly into a paper or technical note.

In [ ]:
def build_latex_table(df: pd.DataFrame, version: str, tag_str: str) -> str:
    """
    Build a LaTeX longtable from the selection-efficiency DataFrame.

    Parameters
    ----------
    df          : the df_table produced in section 4.
    version     : version_run string (used in caption).
    tag_str     : legendtag string (used in caption).

    Returns
    -------
    latex_str   : complete LaTeX code as a string.
    """
    lines = []

    # ── Preamble ──────────────────────────────────────────────────────────────
    lines += [
        r"% ---- AuxTel/Spectractor spectra count & selection efficiency ----",
        r"% Generated automatically by PWV00_SpectraCountTable.ipynb",
        r"",
        r"\begin{longtable}{llcccc}",
        r"\caption{AuxTel/Spectractor spectra count and selection efficiency.\\{}",
        rf"  Run: \texttt{{{version}}} — {tag_str}}}",
        r"\label{tab:spectra_selection_efficiency}\\[4pt]",
        r"\hline\hline",
        r"\multicolumn{1}{c}{\textbf{Epoch}} &",
        r"\multicolumn{1}{c}{\textbf{Filter subset}} &",
        r"\multicolumn{1}{c}{\textbf{Cut level}} &",
        r"\multicolumn{1}{c}{$N_{\rm total}$} &",
        r"\multicolumn{1}{c}{$N_{\rm selected}$} &",
        r"\multicolumn{1}{c}{$\varepsilon$ (\%)} \\\\",
        r"\hline",
        r"\endfirsthead",
        r"",
        r"\multicolumn{6}{c}{\tablename~\thetable{} -- \textit{continued from previous page}} \\\\",
        r"\hline\hline",
        r"\multicolumn{1}{c}{\textbf{Epoch}} &",
        r"\multicolumn{1}{c}{\textbf{Filter subset}} &",
        r"\multicolumn{1}{c}{\textbf{Cut level}} &",
        r"\multicolumn{1}{c}{$N_{\rm total}$} &",
        r"\multicolumn{1}{c}{$N_{\rm selected}$} &",
        r"\multicolumn{1}{c}{$\varepsilon$ (\%)} \\\\",
        r"\hline",
        r"\endhead",
        r"",
        r"\hline",
        r"\multicolumn{6}{r}{\textit{Continued on next page}} \\\\",
        r"\endfoot",
        r"",
        r"\hline\hline",
        r"\endlastfoot",
        r"",
    ]

    # ── Body ──────────────────────────────────────────────────────────────────
    cut_order = ["no cuts", "loose", "standard", "tight"]

    for epoch_label, df_epoch in df.groupby("Epoch", sort=False):
        epoch_tex = epoch_label.replace("\n", " ")  # flatten newline
        first_epoch = True

        for filt_label, df_filt in df_epoch.groupby("Filter subset", sort=False):
            first_filt = True

            # Sort cuts in desired order
            df_filt_sorted = df_filt.set_index("Cut level").reindex(
                [c for c in cut_order if c in df_filt["Cut level"].values]
            ).reset_index()

            n_rows = len(df_filt_sorted)

            for i, row in df_filt_sorted.iterrows():
                cut_tex = row["Cut level"]

                # Epoch: only printed on first row of this epoch block
                epoch_cell = (r"\multirow{" + str(n_rows * len(FILTER_SUBSETS)) + r"}{*}{\textbf{" + epoch_tex + r"}}")
                filt_cell  = (r"\multirow{" + str(n_rows) + r"}{*}{\texttt{" + filt_label + r"}}")

                if first_epoch and first_filt:
                    col1 = epoch_cell
                    col2 = filt_cell
                elif first_filt:
                    col1 = ""
                    col2 = filt_cell
                else:
                    col1 = ""
                    col2 = ""

                line = (
                    f"  {col1} & {col2} & {cut_tex} & "
                    f"{row['N_total']:,} & {row['N_selected']:,} & "
                    f"{row['Efficiency (%)']:.1f} \\\\"
                )
                lines.append(line)

                first_filt  = False
                first_epoch = False

            lines.append(r"  \hline")

        lines.append(r"  \hline")

    lines.append(r"\end{longtable}")
    return "\n".join(lines)


latex_code = build_latex_table(df_table, version_run, tag)
print(latex_code)

---
## 8. Save LaTeX table to file

In [ ]:
latex_output_file = f"{pathdata}/table_spectra_selection_efficiency_{version_run}.tex"

with open(latex_output_file, "w") as f:
    f.write(latex_code)

print(f"LaTeX table saved to: {latex_output_file}")

---
## 9. Summary printout

In [ ]:
print("=" * 80)
print("SUMMARY — AuxTel/Spectractor spectra selection efficiency")
print(f"Run: {version_run}")
print(f"Tag: {tag}")
print("=" * 80)
print(df_table.to_string(index=False))
print("=" * 80)

---
## 10. Notes

- The **collimator** was installed on AuxTel around 2023-09-30. Observations taken
  after that date provide significantly more repeatable D2CCD values and hence
  more reliable PWV measurements.
- The three JSON cut files differ in strictness:
  - **loose** : relaxed thresholds on all parameters (maximises statistics at the
    cost of purity).
  - **standard** : the recommended operating point balancing purity and efficiency.
  - **tight** : strict thresholds (maximises purity at the cost of statistics).
- The *efficiency* $\varepsilon$ is defined as
  $\varepsilon = N_{\rm selected} / N_{\rm total}$ where $N_{\rm total}$
  is the number of spectra in the epoch × filter-subset before any quality cut
  (but after PWV filter selection and removal of rows with missing PWV values).